# Interactive development and recovery notebook
Reviewed 2026-10-05. Output-cleared copy of the current Colab notebook. Choose the new-build or saved-checkpoint recovery path as documented in `docs/reproduction.md`; these are alternative entry points. Scheduled refresh is maintained separately in `scripts/refresh_pipeline.py`. The complete saved checkpoint is `complaints_complete.duckdb`.

In [ ]:
%pip install -q pyarrow

In [ ]:
%pip install -q duckdb dbt-duckdb

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import csv
import io
import zipfile
import pandas as pd

folder = Path("/content/drive/MyDrive/Saved from Chrome")
inventory = []

def add_record(path, member, columns, size):
    inventory.append({
        "file": path.name,
        "csv_inside_zip": member,
        "csv_size_MB": round(size / 1_000_000, 2),
        "columns": len(columns),
        "has_narrative": "Consumer complaint narrative" in columns,
    })

if not folder.is_dir():
    raise FileNotFoundError(f"Folder not found: {folder}")

for path in sorted(folder.rglob("*")):
    if not path.is_file():
        continue

    if path.suffix.lower() == ".csv":
        with path.open("r", encoding="utf-8-sig", newline="") as f:
            columns = next(csv.reader(f))
        add_record(path, None, columns, path.stat().st_size)

    elif path.suffix.lower() == ".zip":
        with zipfile.ZipFile(path) as archive:
            for member in archive.infolist():
                if member.filename.lower().endswith(".csv"):
                    with archive.open(member) as f:
                        reader = csv.reader(
                            io.TextIOWrapper(
                                f, encoding="utf-8-sig", newline=""
                            )
                        )
                        columns = next(reader)
                    add_record(
                        path, member.filename, columns, member.file_size
                    )

inventory_df = pd.DataFrame(inventory)
print(f"CSV entries found: {len(inventory_df)}")
display(inventory_df)

In [ ]:
from pathlib import Path
import zipfile
import pandas as pd

folder = Path("/content/drive/MyDrive/Saved from Chrome")
sources = []
seen = {}
skipped = []

for path in sorted(folder.rglob("CCDB_Export_*.zip")):
    with zipfile.ZipFile(path) as archive:
        for member in archive.infolist():
            if not member.filename.lower().endswith(".csv"):
                continue

            # Identify duplicate copies using ZIP member metadata.
            signature = (
                member.filename,
                member.file_size,
                member.CRC,
            )

            if signature in seen:
                skipped.append({
                    "skipped": path.name,
                    "same_as": seen[signature],
                })
                continue

            seen[signature] = path.name
            sources.append({
                "zip_path": str(path),
                "csv_member": member.filename,
                "csv_size_MB": round(member.file_size / 1_000_000, 2),
            })

source_manifest = pd.DataFrame(sources)

print(f"Selected CSV sources: {len(source_manifest)}")
display(source_manifest)

print("\nDuplicate copies skipped:")
display(pd.DataFrame(skipped))

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import zipfile
import json
import pandas as pd
import pyarrow.parquet as pq

bronze_root = Path("/content/financial_complaint_intelligence/bronze")
bronze_root.mkdir(parents=True, exist_ok=True)

ingested_at = datetime.now(timezone.utc).isoformat()
results = []

for source in source_manifest.to_dict("records"):
    csv_member = source["csv_member"]
    output_dir = bronze_root / Path(csv_member).stem
    marker = output_dir / "_SUCCESS.json"

    # Resume completed archives if this cell is run again.
    if marker.exists():
        results.append(json.loads(marker.read_text()))
        print(f"Already completed: {csv_member}")
        continue

    output_dir.mkdir(parents=True, exist_ok=True)

    # Remove only incomplete generated parts from a previous attempt.
    for part in output_dir.glob("part-*.parquet"):
        part.unlink()

    rows = 0
    narrative_rows = 0

    print(f"\nProcessing: {csv_member}", flush=True)

    with zipfile.ZipFile(source["zip_path"]) as archive:
        with archive.open(csv_member) as csv_file:
            chunks = pd.read_csv(
                csv_file,
                dtype="string",
                keep_default_na=False,
                chunksize=50_000,
            )

            for batch_number, chunk in enumerate(chunks):
                # Preserve source values, including blanks and masked ZIPs.
                chunk["_source_archive"] = Path(source["zip_path"]).name
                chunk["_source_csv"] = csv_member
                chunk["_source_record_number"] = range(
                    rows + 1, rows + len(chunk) + 1
                )
                chunk["_ingested_at"] = ingested_at

                narrative_rows += (
                    chunk["Consumer complaint narrative"]
                    .str.strip()
                    .ne("")
                    .sum()
                )
                rows += len(chunk)

                chunk.to_parquet(
                    output_dir / f"part-{batch_number:05d}.parquet",
                    engine="pyarrow",
                    compression="zstd",
                    index=False,
                )

    # Verify that the Parquet files contain every processed record.
    written_rows = sum(
        pq.ParquetFile(part).metadata.num_rows
        for part in output_dir.glob("part-*.parquet")
    )
    assert written_rows == rows, f"Row count mismatch: {csv_member}"

    result = {
        "source": csv_member,
        "rows": rows,
        "rows_with_narrative": int(narrative_rows),
        "parquet_MB": round(
            sum(p.stat().st_size for p in output_dir.glob("*.parquet"))
            / 1_000_000,
            2,
        ),
    }

    marker.write_text(json.dumps(result, indent=2))
    results.append(result)
    print(f"Completed and verified: {rows:,} records", flush=True)

bronze_summary = pd.DataFrame(results)
display(bronze_summary)

print(f"\nTotal bronze records: {bronze_summary['rows'].sum():,}")

In [ ]:
from pathlib import Path
import shutil
import pandas as pd

project_root = Path("/content/financial_complaint_intelligence")
checkpoint_root = Path(
    "/content/drive/MyDrive/Financial-Complaint-Intelligence/checkpoints"
)
checkpoint_root.mkdir(parents=True, exist_ok=True)

shutil.copytree(
    project_root / "bronze",
    checkpoint_root / "bronze",
    dirs_exist_ok=True
)

bronze_summary.to_csv(
    checkpoint_root / "bronze_summary.csv", index=False
)
source_manifest.to_csv(
    checkpoint_root / "source_manifest.csv", index=False
)

local_parts = list((project_root / "bronze").rglob("part-*.parquet"))
backup_parts = list((checkpoint_root / "bronze").rglob("part-*.parquet"))

assert len(local_parts) == len(backup_parts), "File count mismatch"

for source in local_parts:
    backup = checkpoint_root / source.relative_to(project_root)
    assert backup.stat().st_size == source.stat().st_size, (
        f"Size mismatch: {source.name}"
    )

print(f"Checkpoint saved: {len(backup_parts)} Parquet files")
print(f"Location: {checkpoint_root}")

In [ ]:
import duckdb
from importlib.metadata import version
import subprocess

print("DuckDB:", duckdb.__version__)
print("dbt-core:", version("dbt-core"))
print("dbt-duckdb:", version("dbt-duckdb"))

result = subprocess.run(
    ["dbt", "--version"],
    capture_output=True,
    text=True
)
print(result.stdout)
print(result.stderr)
print("Exit code:", result.returncode)

In [ ]:
from pathlib import Path
import duckdb

project_root = Path("/content/financial_complaint_intelligence")
bronze_root = project_root / "bronze"

# Use the saved checkpoint if Colab has restarted.
if not list(bronze_root.rglob("part-*.parquet")):
    bronze_root = Path(
        "/content/drive/MyDrive/"
        "Financial-Complaint-Intelligence/checkpoints/bronze"
    )

assert list(bronze_root.rglob("part-*.parquet")), "Bronze files not found"

project_root.mkdir(parents=True, exist_ok=True)
temp_root = project_root / "duckdb_temp"
temp_root.mkdir(exist_ok=True)

con = duckdb.connect(str(project_root / "complaints.duckdb"))
con.execute("SET memory_limit = '2GB'")
con.execute("SET threads = 2")
con.execute(f"SET temp_directory = '{temp_root.as_posix()}'")

parquet_pattern = (bronze_root / "*" / "part-*.parquet").as_posix()

con.execute(f"""
CREATE OR REPLACE VIEW bronze_complaints AS
SELECT *
FROM read_parquet('{parquet_pattern}', union_by_name = true)
""")

# Parse dates for profiling without changing the bronze files.
con.execute("""
CREATE OR REPLACE VIEW profile_complaints AS
SELECT *,
    NULLIF(TRIM("Complaint ID"), '') AS complaint_id_checked,
    COALESCE(
        TRY_CAST("Date received" AS DATE),
        CAST(TRY_STRPTIME("Date received", '%m/%d/%Y') AS DATE)
    ) AS received_date_checked
FROM bronze_complaints
""")

print("OVERALL QUALITY")
quality = con.execute("""
SELECT
    COUNT(*) AS raw_records,
    MIN(received_date_checked) AS earliest_date,
    MAX(received_date_checked) AS latest_date,
    COUNT(*) FILTER (
        WHERE complaint_id_checked IS NULL
    ) AS missing_complaint_ids,
    COUNT(*) FILTER (
        WHERE received_date_checked IS NULL
    ) AS missing_or_unparsed_dates,
    COUNT(*) FILTER (
        WHERE COALESCE(
            TRIM("Consumer complaint narrative"), ''
        ) <> ''
    ) AS records_with_narrative
FROM profile_complaints
""").df()
display(quality)

print("RECORDS AND NARRATIVES BY YEAR")
yearly = con.execute("""
SELECT
    EXTRACT(YEAR FROM received_date_checked) AS year,
    COUNT(*) AS raw_records,
    COUNT(*) FILTER (
        WHERE COALESCE(
            TRIM("Consumer complaint narrative"), ''
        ) <> ''
    ) AS records_with_narrative,
    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE COALESCE(
                TRIM("Consumer complaint narrative"), ''
            ) <> ''
        ) / COUNT(*),
        2
    ) AS narrative_coverage_pct
FROM profile_complaints
GROUP BY 1
ORDER BY 1
""").df()
display(yearly)

print("DUPLICATE COMPLAINT IDs")
duplicates = con.execute("""
WITH id_counts AS (
    SELECT complaint_id_checked, COUNT(*) AS occurrences
    FROM profile_complaints
    WHERE complaint_id_checked IS NOT NULL
    GROUP BY complaint_id_checked
)
SELECT
    COUNT(*) AS distinct_nonempty_ids,
    COUNT(*) FILTER (
        WHERE occurrences > 1
    ) AS ids_appearing_multiple_times,
    COALESCE(SUM(occurrences - 1), 0) AS extra_records
FROM id_counts
""").df()
display(duplicates)

# Save the profiling reports locally and in the checkpoint folder.
report_root = project_root / "profiling"
report_root.mkdir(exist_ok=True)

checkpoint_reports = Path(
    "/content/drive/MyDrive/"
    "Financial-Complaint-Intelligence/checkpoints/profiling"
)
checkpoint_reports.mkdir(parents=True, exist_ok=True)

for name, report in [
    ("overall_quality", quality),
    ("yearly_coverage", yearly),
    ("duplicate_ids", duplicates),
]:
    report.to_csv(report_root / f"{name}.csv", index=False)
    report.to_csv(checkpoint_reports / f"{name}.csv", index=False)

print("Profiling complete. Reports saved.")

In [ ]:
fields = [
    "Company", "Product", "Sub-product", "Issue", "Sub-issue",
    "State", "ZIP code", "Tags", "Submitted via",
    "Date sent to company", "Company public response",
    "Company response to consumer", "Timely response?"
]

checks = []
for field in fields:
    checks.append(f"""
        SELECT
            '{field}' AS field,
            COUNT(*) FILTER (
                WHERE COALESCE(TRIM("{field}"), '') = ''
            ) AS blank_records
        FROM bronze_complaints
    """)

print("MISSING VALUES BY FIELD")
display(con.execute(" UNION ALL ".join(checks)).df())

for field in [
    "Tags", "Submitted via",
    "Company response to consumer", "Timely response?"
]:
    print(f"\nVALUES IN: {field}")
    display(con.execute(f"""
        SELECT
            COALESCE(NULLIF(TRIM("{field}"), ''), '[blank]') AS value,
            COUNT(*) AS complaints
        FROM bronze_complaints
        GROUP BY 1
        ORDER BY complaints DESC
        LIMIT 20
    """).df())

In [ ]:
from pathlib import Path
import yaml

project_root = Path("/content/financial_complaint_intelligence")
dbt_root = project_root / "dbt"

for folder in ["models/staging", "models/silver", "models/gold", "tests"]:
    (dbt_root / folder).mkdir(parents=True, exist_ok=True)

def write_file(relative_path, content):
    (dbt_root / relative_path).write_text(content.strip() + "\n")

# Project configuration
project_config = {
    "name": "financial_complaint_intelligence",
    "version": "1.0.0",
    "config-version": 2,
    "profile": "financial_complaint_intelligence",
    "model-paths": ["models"],
    "test-paths": ["tests"],
    "models": {
        "financial_complaint_intelligence": {
            "staging": {
                "+schema": "staging",
                "+materialized": "view",
            },
            "silver": {
                "+schema": "silver",
                "+materialized": "table",
            },
            "gold": {
                "+schema": "gold",
                "+materialized": "table",
            },
        }
    },
}
write_file("dbt_project.yml", yaml.safe_dump(project_config))

# Connect dbt to the database created during profiling.
profile_config = {
    "financial_complaint_intelligence": {
        "target": "dev",
        "outputs": {
            "dev": {
                "type": "duckdb",
                "path": str(project_root / "complaints.duckdb"),
                "schema": "main",
                "threads": 1,
                "settings": {
                    "memory_limit": "2GB",
                    "temp_directory": str(project_root / "duckdb_temp"),
                },
            }
        },
    }
}
write_file("profiles.yml", yaml.safe_dump(profile_config))

# Register the existing bronze view as a dbt source.
sources = {
    "version": 2,
    "sources": [{
        "name": "bronze",
        "schema": "main",
        "tables": [{
            "name": "bronze_complaints",
            "description": "All selected CFPB exports, preserved in Parquet.",
        }],
    }],
}
write_file("models/sources.yml", yaml.safe_dump(sources))

# Standardize column names and convert blank values to NULL.
text_fields = {
    "Complaint ID": "complaint_id",
    "Company": "company_name",
    "Product": "product",
    "Sub-product": "sub_product",
    "Issue": "issue",
    "Sub-issue": "sub_issue",
    "State": "state",
    "ZIP code": "zip_code",
    "Tags": "tags_raw",
    "Submitted via": "submission_channel",
    "Company public response": "company_public_response",
    "Company response to consumer": "company_response",
}

expressions = [
    f"""NULLIF(TRIM("{original}"), '') AS {clean}"""
    for original, clean in text_fields.items()
]

for original, clean in [
    ("Date received", "date_received"),
    ("Date sent to company", "date_sent_to_company"),
]:
    expressions.append(f"""
    COALESCE(
        TRY_CAST("{original}" AS DATE),
        CAST(TRY_STRPTIME("{original}", '%m/%d/%Y') AS DATE)
    ) AS {clean}
    """)

expressions.extend([
    """
    CASE
        WHEN "Timely response?" = 'Yes' THEN TRUE
        WHEN "Timely response?" = 'No' THEN FALSE
        ELSE NULL
    END AS timely_response
    """,
    """
    CASE
        WHEN COALESCE(TRIM("Consumer complaint narrative"), '') = ''
        THEN NULL
        ELSE "Consumer complaint narrative"
    END AS narrative
    """,
    '"Date sent to company" AS date_sent_raw',
    '"Timely response?" AS timely_response_raw',
    "_source_archive",
    "_source_csv",
    "_source_record_number",
    "_ingested_at",
])

staging_sql = (
    "SELECT\n    "
    + ",\n    ".join(expressions)
    + "\nFROM {{ source('bronze', 'bronze_complaints') }}"
)
write_file("models/staging/stg_complaints.sql", staging_sql)

required_columns = [
    "complaint_id", "company_name", "product",
    "submission_channel", "date_received",
    "date_sent_to_company", "timely_response",
]

columns = []
for column in required_columns:
    tests = ["not_null"]
    if column == "complaint_id":
        tests.append("unique")
    columns.append({"name": column, "data_tests": tests})

model_tests = {
    "version": 2,
    "models": [{
        "name": "stg_complaints",
        "description": (
            "Cleaned complaint records. All source rows are retained; "
            "optional blanks become NULL and narrative text is preserved."
        ),
        "columns": columns,
    }],
}
write_file(
    "models/staging/schema.yml",
    yaml.safe_dump(model_tests, sort_keys=False)
)

# A passing dbt test returns zero rows.
write_file("tests/staging_preserves_rows.sql", """
SELECT 'row_count_mismatch' AS failure
WHERE
    (SELECT COUNT(*) FROM {{ ref('stg_complaints') }})
    <>
    (SELECT COUNT(*) FROM {{ source('bronze', 'bronze_complaints') }})
""")

print("dbt project created:", dbt_root)

In [ ]:
import subprocess
import shutil

# Release the notebook's database connection before dbt opens it.
con.close()

command = [
    "dbt", "build",
    "--project-dir", str(dbt_root),
    "--profiles-dir", str(dbt_root),
    "--select", "stg_complaints",
]

result = subprocess.run(command, text=True)
print("dbt exit code:", result.returncode)

# Save the project and its build results to our checkpoint.
shutil.copytree(
    dbt_root,
    Path(
        "/content/drive/MyDrive/"
        "Financial-Complaint-Intelligence/checkpoints/dbt"
    ),
    dirs_exist_ok=True,
)

if result.returncode != 0:
    raise RuntimeError("dbt reported a failure. Share the output above.")

In [ ]:
from pathlib import Path
import yaml

project_root = Path("/content/financial_complaint_intelligence")
dbt_root = project_root / "dbt"

def write_file(relative_path, content):
    destination = dbt_root / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(content.strip() + "\n")

# Stable IDs based on the complete category values.
# JSON encoding distinguishes NULLs and avoids delimiter ambiguity.
def category_key(*columns):
    return f"md5(CAST(to_json(list_value({', '.join(columns)})) AS VARCHAR))"

company_key = category_key("company_name")
product_key = category_key("product", "sub_product")
issue_key = category_key("product", "sub_product", "issue", "sub_issue")
channel_key = category_key("submission_channel")
tag_key = category_key("tag_name")

# Reusable view with the foreign keys.
write_file("models/staging/stg_complaints_keyed.sql", f"""
SELECT *,
    {company_key} AS company_id,
    {product_key} AS product_category_id,
    {issue_key} AS issue_category_id,
    {channel_key} AS submission_channel_id
FROM {{{{ ref('stg_complaints') }}}}
""")

models = {
    "companies": """
        SELECT DISTINCT company_id, company_name
        FROM {{ ref('stg_complaints_keyed') }}
    """,

    "product_categories": """
        SELECT DISTINCT product_category_id, product, sub_product
        FROM {{ ref('stg_complaints_keyed') }}
    """,

    "issue_categories": """
        SELECT DISTINCT
            issue_category_id, product_category_id, issue, sub_issue
        FROM {{ ref('stg_complaints_keyed') }}
    """,

    "submission_channels": """
        SELECT DISTINCT submission_channel_id, submission_channel
        FROM {{ ref('stg_complaints_keyed') }}
    """,

    "complaints": """
        SELECT
            complaint_id,
            company_id,
            product_category_id,
            issue_category_id,
            submission_channel_id,
            date_received,
            date_sent_to_company,
            state,
            zip_code,
            company_public_response,
            company_response,
            timely_response,
            narrative IS NOT NULL AS has_narrative,
            _source_archive,
            _source_csv,
            _source_record_number,
            _ingested_at
        FROM {{ ref('stg_complaints_keyed') }}
    """,

    "complaint_narratives": """
        SELECT complaint_id, narrative
        FROM {{ ref('stg_complaints') }}
        WHERE narrative IS NOT NULL
    """,
}

# Split the comma-separated tags into individual relationships.
write_file("models/staging/stg_complaint_tags.sql", """
SELECT DISTINCT
    complaint_id,
    TRIM(tag_value) AS tag_name
FROM {{ ref('stg_complaints') }},
    UNNEST(string_split(tags_raw, ',')) AS tag_values(tag_value)
WHERE NULLIF(TRIM(tag_value), '') IS NOT NULL
""")

models["tags"] = f"""
SELECT DISTINCT {tag_key} AS tag_id, tag_name
FROM {{{{ ref('stg_complaint_tags') }}}}
"""

models["complaint_tags"] = f"""
SELECT complaint_id, {tag_key} AS tag_id
FROM {{{{ ref('stg_complaint_tags') }}}}
"""

for name, sql in models.items():
    write_file(f"models/silver/{name}.sql", sql)

# Primary key and relationship tests.
primary_keys = {
    "companies": "company_id",
    "product_categories": "product_category_id",
    "issue_categories": "issue_category_id",
    "submission_channels": "submission_channel_id",
    "complaints": "complaint_id",
    "complaint_narratives": "complaint_id",
    "tags": "tag_id",
}

relationships = {
    "complaints": {
        "company_id": ("companies", "company_id"),
        "product_category_id": ("product_categories", "product_category_id"),
        "issue_category_id": ("issue_categories", "issue_category_id"),
        "submission_channel_id": (
            "submission_channels", "submission_channel_id"
        ),
    },
    "issue_categories": {
        "product_category_id": ("product_categories", "product_category_id")
    },
    "complaint_narratives": {
        "complaint_id": ("complaints", "complaint_id")
    },
    "complaint_tags": {
        "complaint_id": ("complaints", "complaint_id"),
        "tag_id": ("tags", "tag_id"),
    },
}

schema_models = []

for name in models:
    column_tests = {}

    if name in primary_keys:
        column_tests[primary_keys[name]] = ["not_null", "unique"]

    for column, (parent, parent_key) in relationships.get(name, {}).items():
        tests = column_tests.setdefault(column, ["not_null"])
        tests.append({
            "relationships": {
                "arguments": {
                    "to": "{{ ref('" + parent + "') }}",
                    "field": parent_key,
                }
            }
        })

    if name == "complaint_narratives":
        column_tests["narrative"] = ["not_null"]

    schema_models.append({
        "name": name,
        "columns": [
            {"name": column, "data_tests": tests}
            for column, tests in column_tests.items()
        ],
    })

write_file(
    "models/silver/schema.yml",
    yaml.safe_dump(
        {"version": 2, "models": schema_models},
        sort_keys=False,
    ),
)

write_file("tests/silver_preserves_complaints.sql", """
SELECT 'complaint_count_mismatch' AS failure
WHERE
    (SELECT COUNT(*) FROM {{ ref('complaints') }})
    <>
    (SELECT COUNT(*) FROM {{ ref('stg_complaints') }})
""")

write_file("tests/silver_preserves_narratives.sql", """
SELECT 'narrative_count_mismatch' AS failure
WHERE
    (SELECT COUNT(*) FROM {{ ref('complaint_narratives') }})
    <>
    (SELECT COUNT(*) FROM {{ ref('stg_complaints') }}
     WHERE narrative IS NOT NULL)
""")

write_file("tests/unique_complaint_tag_pairs.sql", """
SELECT complaint_id, tag_id
FROM {{ ref('complaint_tags') }}
GROUP BY complaint_id, tag_id
HAVING COUNT(*) > 1
""")

write_file("tests/consistent_product_issue.sql", """
SELECT c.complaint_id
FROM {{ ref('complaints') }} c
JOIN {{ ref('issue_categories') }} i
    ON c.issue_category_id = i.issue_category_id
WHERE c.product_category_id <> i.product_category_id
""")

print("Created 8 silver models and their data tests.")

In [ ]:
import subprocess
import duckdb
import shutil
import pandas as pd

result = subprocess.run([
    "dbt", "build",
    "--project-dir", str(dbt_root),
    "--profiles-dir", str(dbt_root),
], text=True)

print("dbt exit code:", result.returncode)

checkpoint_root = Path(
    "/content/drive/MyDrive/"
    "Financial-Complaint-Intelligence/checkpoints"
)

# Save the project and logs, including any failure details.
shutil.copytree(
    dbt_root,
    checkpoint_root / "dbt",
    dirs_exist_ok=True,
)

if result.returncode != 0:
    raise RuntimeError("Build failed. Share the dbt error output above.")

database_path = project_root / "complaints.duckdb"

with duckdb.connect(str(database_path)) as silver_con:
    counts = []
    for table in models:
        row_count = silver_con.execute(
            f'SELECT COUNT(*) FROM main_silver."{table}"'
        ).fetchone()[0]
        counts.append({"table": table, "rows": row_count})

    silver_con.execute("CHECKPOINT")

silver_summary = pd.DataFrame(counts)
display(silver_summary)

silver_summary.to_csv(
    checkpoint_root / "silver_summary.csv",
    index=False,
)

# Save the database containing the built silver tables.
shutil.copy2(
    database_path,
    checkpoint_root / "complaints.duckdb",
)

print("Silver build, tests and checkpoint completed.")

In [ ]:
from pathlib import Path
import yaml

project_root = Path("/content/financial_complaint_intelligence")
dbt_root = project_root / "dbt"

def write_file(relative_path, content):
    destination = dbt_root / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(content.strip() + "\n")

# Each condition produces an integer: 1 if true, otherwise 0.
flag_rules = {
    "timely_response_count": (
        "c.timely_response IS TRUE",
        "Source marks the company response as timely."
    ),
    "not_timely_response_count": (
        "c.timely_response IS FALSE",
        "Source marks the company response as not timely."
    ),
    "known_timeliness_count": (
        "c.timely_response IS NOT NULL",
        "Denominator for the timely response rate."
    ),
    "unknown_timeliness_count": (
        "c.timely_response IS NULL",
        "Timeliness is unavailable."
    ),
    "narrative_count": (
        "c.has_narrative",
        "Published narrative is available."
    ),
    "closed_with_explanation_count": (
        "c.company_response = 'Closed with explanation'",
        "Recorded response outcome is closed with explanation."
    ),
    "monetary_relief_count": (
        "c.company_response = 'Closed with monetary relief'",
        "Recorded outcome includes monetary relief; amount is unknown."
    ),
    "non_monetary_relief_count": (
        "c.company_response = 'Closed with non-monetary relief'",
        "Recorded outcome includes non-monetary relief."
    ),
    "in_progress_count": (
        "c.company_response = 'In progress'",
        "Recorded outcome is in progress in this source export."
    ),
    "untimely_response_outcome_count": (
        "c.company_response = 'Untimely response'",
        "Recorded outcome category; separate from timely_response."
    ),
    "unknown_response_outcome_count": (
        "c.company_response IS NULL",
        "Response outcome is unavailable."
    ),
    "public_response_available_count": (
        "c.company_public_response IS NOT NULL",
        "A company public response category is present."
    ),
    "missing_state_count": (
        "c.state IS NULL",
        "State is unavailable."
    ),
    "missing_zip_count": (
        "c.zip_code IS NULL",
        "ZIP code is unavailable."
    ),
    "missing_issue_count": (
        "i.issue IS NULL",
        "Issue is unavailable."
    ),
    "missing_sub_issue_count": (
        "i.sub_issue IS NULL",
        "Sub-issue is unavailable."
    ),
    "sent_before_received_count": (
        "c.date_sent_to_company < c.date_received",
        "Recorded sending date precedes receipt; investigate date quality."
    ),
}

flag_sql = ",\n    ".join(
    f"CASE WHEN {condition} THEN 1 ELSE 0 END AS {name}"
    for name, (condition, description) in flag_rules.items()
)

write_file("models/gold/complaint_metrics.sql", """
SELECT
    c.*,
    co.company_name,
    p.product,
    p.sub_product,
    i.issue,
    i.sub_issue,
    ch.submission_channel,
    1 AS complaint_count,
    DATE_DIFF(
        'day', c.date_received, c.date_sent_to_company
    ) AS days_to_send_to_company,
    """ + flag_sql + """
FROM {{ ref('complaints') }} c
LEFT JOIN {{ ref('companies') }} co
    ON c.company_id = co.company_id
LEFT JOIN {{ ref('product_categories') }} p
    ON c.product_category_id = p.product_category_id
LEFT JOIN {{ ref('issue_categories') }} i
    ON c.issue_category_id = i.issue_category_id
LEFT JOIN {{ ref('submission_channels') }} ch
    ON c.submission_channel_id = ch.submission_channel_id
""")

# A view avoids storing a second copy of all narrative text.
# This is input for future search indexing, not yet a vector index.
write_file("models/gold/narrative_search_documents.sql", """
{{ config(materialized='view') }}

SELECT
    m.complaint_id,
    m.date_received,
    m.company_id,
    m.company_name,
    m.product,
    m.sub_product,
    m.issue,
    m.sub_issue,
    m.state,
    m.company_response,
    m._source_archive,
    m._source_csv,
    n.narrative
FROM {{ ref('complaint_metrics') }} m
JOIN {{ ref('complaint_narratives') }} n
    ON m.complaint_id = n.complaint_id
""")

metric_columns = [
    {
        "name": "complaint_id",
        "description": "One unique complaint per row.",
        "data_tests": ["not_null", "unique"],
    },
    {
        "name": "complaint_count",
        "description": "Always 1. Sum to calculate complaint volume.",
        "data_tests": ["not_null"],
    },
    {
        "name": "days_to_send_to_company",
        "description": (
            "Days from receipt to sending to company. "
            "Does not measure company response or resolution time."
        ),
    },
]

for name, (_, description) in flag_rules.items():
    metric_columns.append({
        "name": name,
        "description": description,
        "data_tests": ["not_null"],
    })

for column in [
    "date_received", "company_name", "product", "submission_channel"
]:
    metric_columns.append({
        "name": column,
        "data_tests": ["not_null"],
    })

write_file("models/gold/schema.yml", yaml.safe_dump({
    "version": 2,
    "models": [
        {
            "name": "complaint_metrics",
            "description": (
                "Complaint-level business metrics. No date filtering. "
                "Aggregate count flags before calculating rates."
            ),
            "columns": metric_columns,
        },
        {
            "name": "narrative_search_documents",
            "description": (
                "Published narratives joined to business context "
                "for future evidence retrieval."
            ),
            "columns": [
                {
                    "name": "complaint_id",
                    "data_tests": ["not_null", "unique"],
                },
                {
                    "name": "narrative",
                    "data_tests": ["not_null"],
                },
            ],
        },
    ],
}, sort_keys=False))

# Verify that joins preserved every complaint.
write_file("tests/gold_preserves_complaints.sql", """
SELECT 'complaint_count_mismatch' AS failure
WHERE
    (SELECT COUNT(*) FROM {{ ref('complaint_metrics') }})
    <>
    (SELECT COUNT(*) FROM {{ ref('complaints') }})
""")

write_file("tests/gold_preserves_narratives.sql", """
SELECT 'narrative_count_mismatch' AS failure
WHERE
    (SELECT COUNT(*) FROM {{ ref('narrative_search_documents') }})
    <>
    (SELECT COUNT(*) FROM {{ ref('complaint_narratives') }})
""")

invalid_flags = " OR ".join(
    f"{name} NOT IN (0, 1)" for name in flag_rules
)
write_file("tests/gold_flags_are_binary.sql", f"""
SELECT complaint_id
FROM {{{{ ref('complaint_metrics') }}}}
WHERE complaint_count <> 1 OR {invalid_flags}
""")

write_file("tests/gold_flag_consistency.sql", """
SELECT complaint_id
FROM {{ ref('complaint_metrics') }}
WHERE
    timely_response_count + not_timely_response_count
        <> known_timeliness_count
    OR known_timeliness_count + unknown_timeliness_count <> 1
    OR narrative_count <> CASE WHEN has_narrative THEN 1 ELSE 0 END
    OR closed_with_explanation_count
        + monetary_relief_count
        + non_monetary_relief_count
        + in_progress_count
        + untimely_response_outcome_count
        + unknown_response_outcome_count <> 1
""")

# Definitions for the dashboard and future data agent.
write_file("metric_definitions.yml", yaml.safe_dump({
    "grain": "one row per complaint",
    "default_date": "date_received",
    "week_starts": "Monday",
    "complaint_volume": "SUM(complaint_count)",
    "timely_response_rate_pct": (
        "100.0 * SUM(timely_response_count) / "
        "NULLIF(SUM(known_timeliness_count), 0)"
    ),
    "narrative_coverage_pct": (
        "100.0 * SUM(narrative_count) / "
        "NULLIF(SUM(complaint_count), 0)"
    ),
    "limitations": [
        "Metrics describe the downloaded exports, not a live feed.",
        "Response outcomes do not measure customer satisfaction.",
        "Relief categories do not provide financial amounts.",
        "Days to send to company is not resolution time.",
        "No published narrative does not mean no customer problem.",
        "Missing state or ZIP does not imply a missing customer address.",
    ],
}, sort_keys=False))

print("Gold models, flags, metric definitions and tests created.")

In [ ]:
import subprocess
import duckdb
import pandas as pd
import shutil

result = subprocess.run([
    "dbt", "build",
    "--project-dir", str(dbt_root),
    "--profiles-dir", str(dbt_root),
    "--select", "path:models/gold",
], text=True)

print("dbt exit code:", result.returncode)

checkpoint_root = Path(
    "/content/drive/MyDrive/"
    "Financial-Complaint-Intelligence/checkpoints"
)

shutil.copytree(
    dbt_root,
    checkpoint_root / "dbt",
    dirs_exist_ok=True,
)

if result.returncode != 0:
    raise RuntimeError("Gold build failed. Share the error output above.")

database_path = project_root / "complaints.duckdb"

with duckdb.connect(str(database_path)) as gold_con:
    summary = gold_con.execute("""
        SELECT
            COUNT(*) AS complaints,
            SUM(complaint_count) AS complaint_count_total,
            SUM(narrative_count) AS complaints_with_narrative,
            SUM(timely_response_count) AS timely_responses,
            SUM(not_timely_response_count) AS not_timely_responses,
            SUM(known_timeliness_count) AS known_timeliness,
            SUM(unknown_response_outcome_count) AS missing_response_outcomes
        FROM main_gold.complaint_metrics
    """).df()

    gold_con.execute("CHECKPOINT")

display(summary)

summary.to_csv(
    checkpoint_root / "gold_summary.csv",
    index=False,
)

shutil.copy2(
    database_path,
    checkpoint_root / "complaints.duckdb",
)

print("Gold build, tests and checkpoint completed.")

In [ ]:
from pathlib import Path
import yaml

project_root = Path("/content/financial_complaint_intelligence")
dbt_root = project_root / "dbt"

def write_star_model(name, sql):
    destination = dbt_root / "models/star" / f"{name}.sql"
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(
        "{{ config(materialized='table', schema='star') }}\n\n"
        + sql.strip() + "\n"
    )

def stable_key(*columns):
    return (
        "md5(CAST(to_json(list_value("
        + ", ".join(columns)
        + ")) AS VARCHAR))"
    )

geography_key = stable_key("state", "zip_code")
response_key = stable_key(
    "company_response", "company_public_response",
    "CAST(timely_response AS VARCHAR)"
)

# Dimensions connect directly to the fact table.
write_star_model("dim_company", """
SELECT company_id, company_name
FROM {{ ref('companies') }}
""")

write_star_model("dim_product", """
SELECT product_category_id, product, sub_product
FROM {{ ref('product_categories') }}
""")

# Include product context as attributes, without a dimension-to-dimension FK.
write_star_model("dim_issue", """
SELECT
    i.issue_category_id,
    p.product AS issue_product,
    p.sub_product AS issue_sub_product,
    i.issue,
    i.sub_issue
FROM {{ ref('issue_categories') }} i
JOIN {{ ref('product_categories') }} p
    ON i.product_category_id = p.product_category_id
""")

write_star_model("dim_submission_channel", """
SELECT submission_channel_id, submission_channel
FROM {{ ref('submission_channels') }}
""")

write_star_model("dim_geography", f"""
SELECT DISTINCT
    {geography_key} AS geography_id,
    state,
    zip_code
FROM {{{{ ref('complaint_metrics') }}}}
""")

write_star_model("dim_response", f"""
SELECT DISTINCT
    {response_key} AS response_id,
    company_response,
    company_public_response,
    timely_response
FROM {{{{ ref('complaint_metrics') }}}}
""")

# Continuous calendar covering both observed date fields.
write_star_model("dim_date", """
WITH bounds AS (
    SELECT
        LEAST(
            MIN(date_received), MIN(date_sent_to_company)
        ) AS first_date,
        GREATEST(
            MAX(date_received), MAX(date_sent_to_company)
        ) AS last_date
    FROM {{ ref('complaint_metrics') }}
),
calendar AS (
    SELECT CAST(day_value AS DATE) AS calendar_date
    FROM bounds,
    LATERAL generate_series(
        first_date, last_date, INTERVAL '1 day'
    ) AS days(day_value)
)
SELECT
    CAST(STRFTIME(calendar_date, '%Y%m%d') AS INTEGER) AS date_key,
    calendar_date,
    EXTRACT(YEAR FROM calendar_date)::INTEGER AS year,
    EXTRACT(QUARTER FROM calendar_date)::INTEGER AS quarter,
    EXTRACT(MONTH FROM calendar_date)::INTEGER AS month,
    STRFTIME(calendar_date, '%B') AS month_name,
    CAST(DATE_TRUNC('month', calendar_date) AS DATE) AS month_start,
    CAST(DATE_TRUNC('week', calendar_date) AS DATE) AS week_start,
    EXTRACT(ISOYEAR FROM calendar_date)::INTEGER AS iso_year,
    EXTRACT(WEEK FROM calendar_date)::INTEGER AS iso_week,
    EXTRACT(ISODOW FROM calendar_date)::INTEGER AS weekday_number,
    STRFTIME(calendar_date, '%A') AS weekday_name,
    EXTRACT(ISODOW FROM calendar_date) IN (6, 7) AS is_weekend
FROM calendar
""")

# Reuse exactly the metrics already implemented in wide gold.
flag_columns = [
    "complaint_count",
    "timely_response_count",
    "not_timely_response_count",
    "known_timeliness_count",
    "unknown_timeliness_count",
    "narrative_count",
    "closed_with_explanation_count",
    "monetary_relief_count",
    "non_monetary_relief_count",
    "in_progress_count",
    "untimely_response_outcome_count",
    "unknown_response_outcome_count",
    "public_response_available_count",
    "missing_state_count",
    "missing_zip_count",
    "missing_issue_count",
    "missing_sub_issue_count",
    "sent_before_received_count",
]

write_star_model("fact_complaints", f"""
SELECT
    complaint_id,
    company_id,
    product_category_id,
    issue_category_id,
    submission_channel_id,
    {geography_key} AS geography_id,
    {response_key} AS response_id,
    CAST(STRFTIME(date_received, '%Y%m%d') AS INTEGER)
        AS received_date_key,
    CAST(STRFTIME(date_sent_to_company, '%Y%m%d') AS INTEGER)
        AS sent_date_key,
    days_to_send_to_company,
    {", ".join(flag_columns)}
FROM {{{{ ref('complaint_metrics') }}}}
""")

primary_keys = {
    "dim_company": "company_id",
    "dim_product": "product_category_id",
    "dim_issue": "issue_category_id",
    "dim_submission_channel": "submission_channel_id",
    "dim_geography": "geography_id",
    "dim_response": "response_id",
    "dim_date": "date_key",
    "fact_complaints": "complaint_id",
}

fact_relationships = {
    "company_id": ("dim_company", "company_id"),
    "product_category_id": ("dim_product", "product_category_id"),
    "issue_category_id": ("dim_issue", "issue_category_id"),
    "submission_channel_id": (
        "dim_submission_channel", "submission_channel_id"
    ),
    "geography_id": ("dim_geography", "geography_id"),
    "response_id": ("dim_response", "response_id"),
    "received_date_key": ("dim_date", "date_key"),
    "sent_date_key": ("dim_date", "date_key"),
    "complaint_id": ("complaint_metrics", "complaint_id"),
}

schema_models = []
for name, primary_key in primary_keys.items():
    columns = [{
        "name": primary_key,
        "data_tests": ["not_null", "unique"],
    }]

    if name == "fact_complaints":
        for column, (parent, parent_key) in fact_relationships.items():
            relationship_test = {
                "relationships": {
                    "arguments": {
                        "to": "{{ ref('" + parent + "') }}",
                        "field": parent_key,
                    }
                }
            }
            if column == primary_key:
                columns[0]["data_tests"].append(relationship_test)
            else:
                columns.append({
                    "name": column,
                    "data_tests": ["not_null", relationship_test],
                })

        columns.extend(
            {"name": column, "data_tests": ["not_null"]}
            for column in flag_columns
        )

    schema_models.append({"name": name, "columns": columns})

(dbt_root / "models/star/schema.yml").write_text(
    yaml.safe_dump(
        {"version": 2, "models": schema_models},
        sort_keys=False
    )
)

# Reconcile row counts and every metric total against existing gold.
aggregate_expressions = ",\n".join(
    f"SUM({column}) AS {column}"
    for column in flag_columns
)
differences = " OR ".join(
    f"s.{column} IS DISTINCT FROM w.{column}"
    for column in flag_columns
)

(dbt_root / "tests/star_matches_wide_gold.sql").write_text(f"""
WITH star_totals AS (
    SELECT COUNT(*) AS rows, {aggregate_expressions}
    FROM {{{{ ref('fact_complaints') }}}}
),
wide_totals AS (
    SELECT COUNT(*) AS rows, {aggregate_expressions}
    FROM {{{{ ref('complaint_metrics') }}}}
)
SELECT 'star_wide_mismatch' AS failure
FROM star_totals s CROSS JOIN wide_totals w
WHERE s.rows <> w.rows OR {differences}
""")

print("Added 7 dimensions, 1 fact table and reconciliation tests.")
print("Existing gold model files were not changed.")

In [ ]:
import subprocess
import duckdb
import pandas as pd
import shutil

result = subprocess.run([
    "dbt", "build",
    "--project-dir", str(dbt_root),
    "--profiles-dir", str(dbt_root),
    "--select", "path:models/star",
], text=True)

print("dbt exit code:", result.returncode)

checkpoint_root = Path(
    "/content/drive/MyDrive/"
    "Financial-Complaint-Intelligence/checkpoints"
)

shutil.copytree(
    dbt_root,
    checkpoint_root / "dbt",
    dirs_exist_ok=True,
)

if result.returncode != 0:
    raise RuntimeError("Star build failed. Share the error output above.")

database_path = project_root / "complaints.duckdb"

with duckdb.connect(str(database_path)) as star_con:
    counts = []
    for table in primary_keys:
        rows = star_con.execute(
            f'SELECT COUNT(*) FROM main_star."{table}"'
        ).fetchone()[0]
        counts.append({"table": table, "rows": rows})

    star_con.execute("CHECKPOINT")

star_summary = pd.DataFrame(counts)
display(star_summary)

star_summary.to_csv(
    checkpoint_root / "star_summary.csv",
    index=False,
)

shutil.copy2(
    database_path,
    checkpoint_root / "complaints.duckdb",
)

print("Star schema built, tested and saved alongside existing gold.")

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

checkpoint = Path(
    "/content/drive/MyDrive/"
    "Financial-Complaint-Intelligence/checkpoints"
)

for name in [
    "complaints.duckdb",
    "silver_summary.csv",
    "gold_summary.csv",
    "star_summary.csv",
]:
    path = checkpoint / name
    if path.exists():
        print(f"{name}: FOUND — {path.stat().st_size / 1024**2:,.2f} MB")
    else:
        print(f"{name}: MISSING")

In [ ]:
import duckdb
import pandas as pd

with duckdb.connect(str(database), read_only=True) as con:
    objects = con.execute("""
        SELECT table_schema, table_name, table_type
        FROM information_schema.tables
        WHERE table_schema NOT IN (
            'information_schema', 'pg_catalog'
        )
        ORDER BY table_schema, table_name
    """).fetchdf()

    print("ALL DATABASE OBJECTS")
    print(objects.to_string(index=False))

    silver_count = con.execute("""
        SELECT COUNT(*) FROM main_silver.complaints
    """).fetchone()[0]
    print(f"\nSilver complaints: {silver_count:,}")

for name in [
    "silver_summary.csv",
    "gold_summary.csv",
    "star_summary.csv",
]:
    print(f"\n{name}")
    path = checkpoint / name
    if path.exists() and path.stat().st_size:
        print(pd.read_csv(path).to_string(index=False))
    else:
        print("Missing or empty")

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

repo = Path("/content/fci_recovery_repo")

if not repo.exists():
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/Satyavathi-Gunturi/"
        "Financial-Complaint-Intelligence.git",
        str(repo),
    ], check=True)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", str(repo / "requirements.txt"),
], check=True)

shutil.copy2(
    repo / "dbt/profiles.example.yml",
    repo / "dbt/profiles.yml",
)

temp = database.parent / "duckdb_temp"
temp.mkdir(exist_ok=True)

env = os.environ.copy()
env["FCI_DATABASE_PATH"] = str(database)
env["FCI_TEMP_DIRECTORY"] = str(temp)

result = subprocess.run([
    "dbt", "build",
    "--project-dir", str(repo / "dbt"),
    "--profiles-dir", str(repo / "dbt"),
    "--select", "path:models/gold", "path:models/star",
    "--indirect-selection", "cautious",
], env=env)

print(f"\ndbt exit code: {result.returncode}")

if result.returncode != 0:
    raise RuntimeError(
        "Recovery build failed. Keep the output for diagnosis."
    )

print("Gold and star recovery build and selected tests passed.")

In [ ]:
import duckdb
import shutil

with duckdb.connect(str(database), read_only=True) as con:
    counts = {
        "silver": con.execute(
            "SELECT COUNT(*) FROM main_silver.complaints"
        ).fetchone()[0],
        "wide_gold": con.execute(
            "SELECT COUNT(*) FROM main_gold.complaint_metrics"
        ).fetchone()[0],
        "star": con.execute(
            "SELECT COUNT(*) FROM main_star.fact_complaints"
        ).fetchone()[0],
    }

print(counts)
assert all(n == 14_482_997 for n in counts.values()), \
    "Count mismatch: checkpoint not saved."

# Flush database changes before making the backup.
with duckdb.connect(str(database)) as con:
    con.execute("CHECKPOINT")

backup = checkpoint / "complaints_complete.duckdb"
shutil.copy2(database, backup)

# Verify the saved copy contains both gold structures.
with duckdb.connect(str(backup), read_only=True) as con:
    for table in [
        "main_gold.complaint_metrics",
        "main_star.fact_complaints",
    ]:
        count = con.execute(
            f"SELECT COUNT(*) FROM {table}"
        ).fetchone()[0]
        assert count == 14_482_997, f"Backup mismatch: {table}"
        print(f"Saved {table}: {count:,} rows")

print(f"\nComplete checkpoint saved: {backup.name}")
print(f"Size: {backup.stat().st_size / 1024**3:.2f} GB")

In [ ]:
with duckdb.connect(str(database), read_only=True) as con:
    columns = con.execute("""
        DESCRIBE main_gold.complaint_metrics
    """).fetchdf()

    coverage = con.execute("""
        SELECT
            MIN(date_received) AS first_date,
            MAX(date_received) AS last_date,
            COUNT(DISTINCT date_received) AS distinct_dates,
            COUNT(DISTINCT company_id) AS companies,
            COUNT(DISTINCT product_category_id) AS product_categories,
            COUNT(DISTINCT issue_category_id) AS issue_categories
        FROM main_gold.complaint_metrics
    """).fetchdf()

print("GOLD FIELDS")
print(columns[["column_name", "column_type"]].to_string(index=False))
print("\nCOVERAGE")
print(coverage.to_string(index=False))

In [ ]:
import duckdb
import shutil

export = database.parent / "dashboard_daily_company_product.parquet"

with duckdb.connect(str(database), read_only=True) as con:
    columns = con.execute(
        "DESCRIBE main_gold.complaint_metrics"
    ).fetchdf()["column_name"].tolist()

    # Use the existing additive flags, including complaint_count.
    metrics = [c for c in columns if c.endswith("_count")]
    sums = ",\n".join(f'SUM("{c}") AS "{c}"' for c in metrics)

    con.execute(f"""
        COPY (
            SELECT
                date_received,
                company_id,
                company_name,
                product_category_id,
                product,
                sub_product,
                {sums}
            FROM main_gold.complaint_metrics
            GROUP BY
                date_received, company_id, company_name,
                product_category_id, product, sub_product
        )
        TO '{export}'
        (FORMAT PARQUET, COMPRESSION ZSTD)
    """)

    original = con.execute(f"""
        SELECT {sums} FROM main_gold.complaint_metrics
    """).fetchone()

    exported = con.execute(f"""
        SELECT {sums} FROM read_parquet('{export}')
    """).fetchone()

    assert original == exported, "Metric reconciliation failed."

    rows = con.execute(f"""
        SELECT COUNT(*) FROM read_parquet('{export}')
    """).fetchone()[0]

saved_export = checkpoint / export.name
shutil.copy2(export, saved_export)

print(f"Dashboard rows: {rows:,}")
print(f"Parquet size: {export.stat().st_size / 1024**2:,.2f} MB")
print(f"All {len(metrics)} metric totals reconcile.")
print(f"Saved: {saved_export.name}")

In [ ]:
from google.colab import files

files.download(str(saved_export))

In [ ]:
from pathlib import Path
import duckdb
import shutil

export_dir = database.parent / "dashboard_exports"
export_dir.mkdir(exist_ok=True)

saved_dir = checkpoint / "dashboard_exports"
saved_dir.mkdir(exist_ok=True)

datasets = {
    "dashboard_issues.parquet": ["issue", "sub_issue"],
    "dashboard_geography.parquet": ["state"],
    "dashboard_channels.parquet": ["submission_channel"],
}

with duckdb.connect(str(database), read_only=True) as con:
    con.execute("SET memory_limit='2GB'")

    columns = con.execute(
        "DESCRIBE main_gold.complaint_metrics"
    ).fetchdf()["column_name"].tolist()

    metrics = [c for c in columns if c.endswith("_count")]
    sums = ", ".join(f'SUM("{c}") AS "{c}"' for c in metrics)

    expected = con.execute(
        f"SELECT {sums} FROM main_gold.complaint_metrics"
    ).fetchone()

    base = [
        "date_received",
        "company_id", "company_name",
        "product_category_id", "product", "sub_product",
    ]

    for filename, extra_columns in datasets.items():
        fields = ", ".join(
            f'"{c}"' for c in base + extra_columns
        )
        output = export_dir / filename

        print(f"Building {filename}...", flush=True)

        con.execute(f"""
            COPY (
                SELECT {fields}, {sums}
                FROM main_gold.complaint_metrics
                GROUP BY ALL
            )
            TO '{output}'
            (FORMAT PARQUET, COMPRESSION ZSTD)
        """)

        actual = con.execute(f"""
            SELECT {sums}
            FROM read_parquet('{output}')
        """).fetchone()

        assert actual == expected, (
            f"Metric reconciliation failed: {filename}"
        )

        rows = con.execute(f"""
            SELECT COUNT(*) FROM read_parquet('{output}')
        """).fetchone()[0]

        shutil.copy2(output, saved_dir / filename)

        print(
            f"{filename}: {rows:,} rows | "
            f"{output.stat().st_size / 1024**2:.2f} MB | "
            f"all {len(metrics)} metric totals match",
            flush=True,
        )

print("\nAll three exports verified and saved.")

In [ ]:
from google.colab import files
import shutil

archive = shutil.make_archive(
    "/content/dashboard_exports",
    "zip",
    root_dir=str(export_dir),
)
files.download(archive)

In [ ]:
from google.colab import files

folder = (
    "/content/drive/MyDrive/"
    "Financial-Complaint-Intelligence/checkpoints/dashboard_exports"
)

files.download(f"{folder}/dashboard_issues.parquet")

In [ ]:
files.download(f"{folder}/dashboard_geography.parquet")

In [ ]:
files.download(f"{folder}/dashboard_channels.parquet")